In [1]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


BASE_DIR       = r"c:/Users/hpc/Downloads/ClinicApp-main/ClinicAI/notebooks"
MODEL_SAVE_DIR = r"c:/Users/hpc/Downloads/ClinicApp-main/ClinicAI/models"


# -------------------------------------------------------------------
# Dataset configs
# -------------------------------------------------------------------

diabetes_config = {
    "path": os.path.join(BASE_DIR, "diabetes_prediction_dataset.csv"),
    "target": "diabetes",
    "features": [
        "HbA1c_level", "blood_glucose_level", "age", "bmi",
        "smoking_history", "hypertension", "gender", "heart_disease"
    ]
}

heart_config = {
    "path": os.path.join(BASE_DIR, "heart_2022_no_nans.csv"),
    "target": "HadHeartAttack",
    "features": [
        "HadAngina", "ChestScan", "HadStroke", "DifficultyWalking",
        "HadDiabetes", "GeneralHealth", "HadArthritis", "PneumoVaxEver",
        "RemovedTeeth", "AgeCategory", "SmokerStatus", "BMI",
        "HadKidneyDisease", "HadCOPD"
    ]
}

kidney_config = {
    "path": os.path.join(BASE_DIR, "kidney_disease.csv"),
    "target": "classification",
    "features": [
        "age", "bp", "sg", "al", "su",
        "rbc", "pc", "pcc", "ba",
        "bgr", "bu", "sc", "sod", "pot",
        "hemo", "pcv", "wc", "rc",
        "htn", "dm", "cad", "appet", "pe", "ane"
    ]
}

DATASETS = {
    "diabetes": diabetes_config,
    "heart":    heart_config,
    "kidney":   kidney_config
}


# -------------------------------------------------------------------
# Models
# -------------------------------------------------------------------

def make_xgb():
    return CalibratedClassifierCV(
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.7,
            scale_pos_weight=1,
            eval_metric="logloss",
            tree_method="hist",
            random_state=42
        ),
        method="isotonic",
        cv=3
    )


# -------------------------------------------------------------------
# Data cleaning
# -------------------------------------------------------------------

def fix_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    return np.nan if x in ["?", "", "nan", "none"] else x

def to_binary(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return 1 if float(v) != 0 else 0
    s = str(v).strip().lower()
    if s in {"0", "no", "negative", "normal", "false", "notckd"}: return 0
    if s in {"1", "yes", "positive", "abnormal", "true", "ckd"}:  return 1
    return 0 if "not" in s else 1

def prepare_diabetes(df):
    df = df.copy()
    df["smoking_history"] = df["smoking_history"].replace("No Info", np.nan)
    return df

def prepare_kidney(df):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    for col in df.select_dtypes("object").columns:
        df[col] = df[col].apply(fix_value)

    df["classification"] = df["classification"].replace("ckd\t", "ckd")

    for col in ["pcv", "wc", "rc"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "id" in df.columns:
        df = df.drop(columns=["id"])

    return df

def prepare_data(df, disease):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    if disease == "diabetes": return prepare_diabetes(df)
    if disease == "kidney":   return prepare_kidney(df)
    return df


# -------------------------------------------------------------------
# Preprocessor
# -------------------------------------------------------------------

def build_preprocessor(X):
    cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])


# -------------------------------------------------------------------
# Training
# -------------------------------------------------------------------

def train_model(df, feature_cols, target_col, disease):
    df = prepare_data(df, disease)
    df = df[feature_cols + [target_col]].copy()

    X = df[feature_cols].copy()
    y = df[target_col].apply(to_binary)

    X = X[y.notna()]
    y = y[y.notna()].astype(int)

    print(f"\nClass distribution:\n{y.value_counts()}\n")

    preprocessor = build_preprocessor(X)

    if disease == "kidney":
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier",   make_xgb())
        ])
    else:
        model = ImbPipeline([
            ("preprocessor", preprocessor),
            ("smote",        SMOTE(sampling_strategy=0.6, random_state=42)),
            ("classifier",   make_xgb())
        ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"\n{disease.upper()} \u2014 Test Results")
    print(classification_report(y_test, y_pred))

    return model, X


# -------------------------------------------------------------------
# Train all & save as PKL
# -------------------------------------------------------------------

for disease_name, config in DATASETS.items():
    print(f"\n{'='*50}")
    print(f"Loading {disease_name} data and training model...")
    print(f"{'='*50}")
    df = pd.read_csv(config["path"])
    model, X = train_model(df, config["features"], config["target"], disease_name)

    save_path = os.path.join(MODEL_SAVE_DIR, f"{disease_name}_model.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(model, f)
    print(f"\u2713 {disease_name.capitalize()} model saved to {save_path}")

print("\n\u2713 ALL MODELS TRAINED AND SAVED.")


Loading diabetes data and training model...

Class distribution:
diabetes
0    91500
1     8500
Name: count, dtype: int64



C:\Users\hpc\AppData\Local\Temp\ipykernel_2856\2846921783.py:144: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()



DIABETES — Test Results
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     18300
           1       0.87      0.73      0.80      1700

    accuracy                           0.97     20000
   macro avg       0.93      0.86      0.89     20000
weighted avg       0.97      0.97      0.97     20000

✓ Diabetes model saved to c:/Users/hpc/Downloads/ClinicApp-main/ClinicAI/models\diabetes_model.pkl

Loading heart data and training model...

Class distribution:
HadHeartAttack
0    232587
1     13435
Name: count, dtype: int64



C:\Users\hpc\AppData\Local\Temp\ipykernel_2856\2846921783.py:144: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()



HEART — Test Results
              precision    recall  f1-score   support

           0       0.97      0.96      0.97     46518
           1       0.44      0.50      0.47      2687

    accuracy                           0.94     49205
   macro avg       0.71      0.73      0.72     49205
weighted avg       0.94      0.94      0.94     49205

✓ Heart model saved to c:/Users/hpc/Downloads/ClinicApp-main/ClinicAI/models\heart_model.pkl

Loading kidney data and training model...

Class distribution:
classification
1    250
0    150
Name: count, dtype: int64



C:\Users\hpc\AppData\Local\Temp\ipykernel_2856\2846921783.py:118: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object").columns:
C:\Users\hpc\AppData\Local\Temp\ipykernel_2856\2846921783.py:144: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-


KIDNEY — Test Results
              precision    recall  f1-score   support

           0       0.85      0.97      0.91        30
           1       0.98      0.90      0.94        50

    accuracy                           0.93        80
   macro avg       0.92      0.93      0.92        80
weighted avg       0.93      0.93      0.93        80

✓ Kidney model saved to c:/Users/hpc/Downloads/ClinicApp-main/ClinicAI/models\kidney_model.pkl

✓ ALL MODELS TRAINED AND SAVED.
